Filen bygger en RAG chatbot som är specialiserad på hudvård med tips och rekommendationer från kicks.se. Den innehåller metadata kring 3 olika hudyper (fet, torr och kombinerad) och har färdiga dokument kring produkter och tips anpassade efter slutanvändarens input från en Streamlit-app. Chatboten får endast ge rekommendationer och tips från inladdade dokument och använder sedan OpenAIs LLM för att chatta med slutanvändaren. Alla txt-dokument som laddas in delas in i hudtyp och kategorier utifrån sina filnamn, varpå det är viktigt att följa namnstandard när txt-filer döps.
Nedan specificeras vilka tillägg som används till vad:

langchain: Själva "ramverket" som binder ihop textfiler från kicks.se med AI-modellen.

langchain-openai: Den specifika bryggan för att prata med OpenAIs modeller.

chromadb: En Vector Database. Det är här alla meningar sparas som sökbara vektorer (siffror).

In [ ]:
# En kod för att rensa tidigare inläsningar om något skall uppdateras, för att undvika att det krockar med gammal data
base_folder = "." # Hänvisar till den lokala mappen
all_documents = [] # Skapar en tom lista med alla dokument som laddas in

In [ ]:
# Alla imports som behövs för att köra samtliga kodblock
import os
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:

# Skapar en tom behållare för att samla in all textdata innan den bearbetas
base_folder = "." 
all_documents = [] 

# Ladda in alla txt-filer och används filnamnen för att tagga filerna så att RAG enkelt kan hitta
for root, dirs, files in os.walk(base_folder):
    for filename in files:
        if filename.endswith(".txt"):
            file_path = os.path.join(root, filename) # här skapas sökväg baserat på filnamn
            
            try: # Testar att läsa in filer med UTF-8-kodning, och hoppar vidare till nästa fil om någon fil är skadad utan att krasha
                loader = TextLoader(file_path, encoding='utf-8')
                loaded_docs = loader.load()
                
                for doc in loaded_docs: # Skapar variabler med små bokstäver så att inga ord missas pga stora/små bokstäver i filnamnet
                    file_lower = filename.lower()
                    path_lower = file_path.lower()
                    
                    # Delar in i olika hudtyper och sparar som metadata utifrån filnamn
                    if "torr-hud" in path_lower or "torr-hud" in file_lower:
                        doc.metadata["skin_type"] = "torr-hud"
                    elif "fet-hud" in path_lower or "fet-hud" in file_lower:
                        doc.metadata["skin_type"] = "fet-hud"
                    elif "kombinerad-hud" in path_lower or "kombinerad-hud" in file_lower:
                        doc.metadata["skin_type"] = "kombinerad-hud"
                    else:
                        doc.metadata["skin_type"] = "allmänt"
                    
                    # Delar upp i kategorier ch sparar som metadata utifrån filnamn
                    if "rengöring" in file_lower or "rengoring" in file_lower:
                        doc.metadata["category"] = "rengöring"
                    elif "serum" in file_lower:
                        doc.metadata["category"] = "serum"
                    elif "ansiktskräm" in file_lower or "ansiktskram" in file_lower or "lotion" in file_lower:
                        doc.metadata["category"] = "ansiktskräm"
                    else:
                        doc.metadata["category"] = "allmänt"
                    
                    # Sparar källan för spårbarhet
                    doc.metadata["source"] = filename
                    all_documents.append(doc)
            except Exception as e:
                print(f"Kunde inte ladda {filename}: {e}") # Felhantering om filen inte kan laddas in och sparas enligt ovan

# Delar upp txt-dokumenten i chunks. Recurive delar upp texten vid naturliga tecken som radbrytningar eller punkter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
final_chunks = text_splitter.split_documents(all_documents)

print(f"KLART! Totalt antal chunks skapade: {len(final_chunks)}")

KLART! Totalt antal chunks skapade: 49


Ovan ser korrekt ut med 49 skapade chunks. Nu går jag vidare med att skapa embeddings som "fångar" kontexten av alla chunks genom att omvandla alla textstycken (chunks) till matematiska vektorer (listor med siffror).

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

# Initierar embeddings och använder nyckel mot OpenAIs Embedding modell
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small", 
    api_key=st.secrets["OPENAI_API_KEY"] # API nyckel från OpenAI
)

# Skapar ett vektorlager med Chroma. Det är som en databas med alla textfiler, lagrade som chunks, som blir sökbara för RAG
vectorstore = Chroma.from_documents(
    documents=final_chunks, # Detta är alla sparade chunks
    embedding=embeddings,
    persist_directory="./db_skincare_v3" # Vektorlagret sparas som en lokalmapp
)

print(f"Nytt vektorlager skapat med {len(final_chunks)} taggade textbitar!") # För att verifiera

# Testa att söka efter rengöring för se att det RAG fungerar som det är tänkt
filtered_results = vectorstore.similarity_search(
    "Vilken tvätt passar mig?", 
    k=2,# Hämtar de 2 mest relevanta textbitarna som matchar sökningen från databasen
    filter={"$and": [{"skin_type": "fet-hud"}, {"category": "rengöring"}]}
)

print("\n--- Resultat av test (Endast rengöring för fet hud) ---")
for r in filtered_results:
    print(f"Källa: {r.metadata['source']}")
    print(f"Innehåll: {r.page_content[:150]}...")
    print("-" * 10)

Nytt smart vektorlager skapat med 49 taggade textbitar!

--- Resultat med filter (Endast rengöring för fet hud) ---
Källa: rengöring-fet-hud.txt
Innehåll: Pris: 280 kr.

Länk: https://www.kicks.se/kiehls-rare-earth-deep-pore-daily-cleanser-150-ml

2. COSRX - Low pH Good Morning Gel Cleanser

Typ: Mild re...
----------
Källa: rengöring-fet-hud.txt
Innehåll: Tips på ansiktsrengöring för fet hy

För en fet hudtyp behöver vi rengöringar som effektivt löser upp överskottsfett och rengör porerna på djupet utan...
----------


Skapar själva Hjärnan (LLM-kedjan) som pratar med databasen och sammanfattar svaren åt användaren.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

# TEST (Här kan man ändra för att testa olika val) 
user_question = "Jag letar efter ett bra serum, vad rekommenderar ni?"
selected_skin_type = "fet-hud"  # "torr-hud", "fet-hud", "kombinerad-hud"
selected_category = "serum"     # "rengöring", "serum", "ansiktskräm"

# Kopplar upp mot OpenAIs LLM
llm = ChatOpenAI(model="gpt-4o", api_key=st.secrets["OPENAI_API_KEY"])

# Skapar en system prompt till RAG med instruktioner kring syfte
system_prompt = (
    "Du är en hudvårdsexpert på Kicks. "
    "Använd kontexten nedan för att svara på frågan. "
    "VIKTIGT: Om samma produkt förekommer flera gånger i kontexten, presentera den bara EN gång. "
    "Gör svaret koncist och pedagogiskt. Inkludera alltid pris och länk."
    "\n\n"
    "{context}" # Här skickas mina dokument med till OpenAIs LLM
)

# Skapar en mall som använder slutanvändarens prompt som input
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

# Kopplar ihop med skapat vektorlager
embeddings = OpenAIEmbeddings(model="text-embedding-3-small", api_key=st.secrets["OPENAI_API_KEY"])

vectorstore = Chroma(
    persist_directory="./db_skincare_v3", 
    embedding_function=embeddings
)

# Dynamisk retriever med AND-filter (Hittar exakt rätt kombination)
retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 5, # Hämtar 5 bästa matcherna
        "filter": {
            "$and": [

                {"skin_type": selected_skin_type},
                {"category": selected_category}
            ]
        }
    }
)

# Skapar en kedja för en naturlig konversation och använder endast inlästa dokument som underlag för svar
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

# Kör sökningen från användarens prompt till RAG 
response = rag_chain.invoke({"input": user_question})

print(f"\n--- TEST-INFO ---")
print(f"Filter: {selected_skin_type} + {selected_category}")
print(f"\n--- CHATBOTENS SVAR ---")
print(response["answer"])


--- TEST-INFO ---
Filter: fet-hud + serum

--- CHATBOTENS SVAR ---
Jag rekommenderar **The Ordinary - Niacinamide 10% + Zinc 1%** serum. Det är ett behandlande serum som balanserar talgproduktionen, jämnar ut hudtonen och har lugnande samt porsammandragande effekter, vilket är perfekt för fet hud. 

Pris: 79 kr.  

Länk: [The Ordinary - Niacinamide 10% + Zinc 1%](https://www.kicks.se/the-ordinary-niacinamide-10-zinc-1-30-ml)


Jag är nöjd med ovan output för torr hud, och vill nu gå vidare med även andra hudtyper.